In [1]:
import pandas as pd
import plotly.express as px
from scipy.stats import pearsonr
import numpy as np
import plotly.figure_factory as ff
# kaleido
# nbformat

In [2]:
df = pd.read_csv('corr.csv').drop(columns=["Added By", "Notes", "Abrivation"])

In [3]:
df.head()

,Model,Method Used,Test Dataset,Source,Accuracy,Domain
0,T5-780M,SFT on Ab-Train (UniADILR-PSy),Ab-test/De-test/In-test (UniADILR-PSy),UniADILR (Table 2) \nCOLING 2024,100.0,Abduction
1,T5-780M,SFT on Complement-train (UniADILR-PSy),Ab-test/De-test/In-test (UniADILR-PSy),UniADILR (Table 2) \nCOLING 2024,55.7,Abduction
2,T5-3B,SFT on Ab-Train (UniADILR-PSy),Ab-test/De-test/In-test (UniADILR-PSy),UniADILR (Table 2) \nCOLING 2024,100.0,Abduction
3,T5-3B,SFT on Complement-train (UniADILR-PSy),Ab-test/De-test/In-test (UniADILR-PSy),UniADILR (Table 2) \nCOLING 2024,59.2,Abduction
4,T5-3B,SFT on Ab-Train (UniADILR-PSy),UniADILR-HGc-Ab UniADILR-HGc-De UniADILR-HGc-In,UniADILR (Figure 3) \nCOLING 2024,53.4,Abduction


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276 entries, 0 to 275
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Model         276 non-null    object 
 1   Method Used   276 non-null    object 
 2   Test Dataset  276 non-null    object 
 3   Source        276 non-null    object 
 4   Accuracy      276 non-null    float64
 5   Domain        276 non-null    object 
dtypes: float64(1), object(5)
memory usage: 13.1+ KB


In [5]:
df.Model.value_counts()

Model
GPT-4               69
GPT-4o              30
Claude V3 Sonnet    30
 LLaMA3.1-Chat      30
Mistral-Ins-v0.3    30
Claude-3.5          30
LLaMA2-70B          12
GPT-3.5             12
LLaMA2-7B            9
LLaMA2-13B           9
T5-3B                9
T5-780M              6
Name: count, dtype: int64

In [6]:
# Define a function to categorize models into families
def get_model_family(model_name):
    if 'GPT' in model_name or 'gpt' in model_name:
        return 'GPT'
    elif 'Claude' in model_name:
        return 'Claude'
    elif 'LLaMA' in model_name:
        return 'LLaMA'
    elif 'T5' in model_name:
        return 'T5'
    elif 'Mistral' in model_name:
        return 'Mistral'
    else:
        return 'Other'

# Split the dataframe by domain
df_abduction = df[df['Domain'] == 'Abduction'].copy()
df_deduction = df[df['Domain'] == 'Deduction'].copy()
df_induction = df[df['Domain'] == 'Induction'].copy()
df_mix = df[df['Domain'] == 'MIX (A-D-I)'].copy()

merged_ded = pd.merge(df_abduction, df_deduction, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Deduction'))
merged_ind = pd.merge(df_abduction, df_induction, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Induction'))
merged_mix = pd.merge(df_abduction, df_mix, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Mix'))

In [7]:
# --- Prepare Data for Combined Plot ---

# Standardize Deduction data
df1 = merged_ded.rename(columns={'Accuracy_Deduction': 'Accuracy', 'Test Dataset_Deduction': 'Test Dataset_Other'})
df1['Domain_Other'] = 'Deduction'

# Standardize Induction data
df2 = merged_ind.rename(columns={'Accuracy_Induction': 'Accuracy', 'Test Dataset_Induction': 'Test Dataset_Other'})
df2['Domain_Other'] = 'Induction'

# Standardize Mix data
df3 = merged_mix.rename(columns={'Accuracy_Mix': 'Accuracy', 'Test Dataset_Mix': 'Test Dataset_Other'})
df3['Domain_Other'] = 'Mix'

# Combine all dataframes
common_cols = ['Accuracy_Abduction', 'Accuracy', 'Test Dataset_Abduction', 'Test Dataset_Other', 'Domain_Other', 'Model', 'Method Used', 'Source']
combined_df = pd.concat([
    df1[common_cols],
    df2[common_cols],
    df3[common_cols]
], ignore_index=True)

# Create the color group label based ONLY on the domain pair
combined_df['Domain Pair'] = 'Abduction vs. ' + combined_df['Domain_Other']

In [8]:
domains = ['Abduction', 'Deduction', 'Induction', 'MIX (A-D-I)']


# --- Prepare data for the distribution plot ---
# Create a list of arrays, where each array contains the accuracies for one domain.
hist_data = [df[df['Domain'] == domain]['Accuracy'] for domain in domains]


# --- Create the plot with KDE curves only ---
# The create_distplot function can hide the histogram by setting show_hist=False
fig_kde_only = ff.create_distplot(
    hist_data,
    group_labels=domains,
    show_hist=False, # This parameter hides the histogram bars
    show_rug=False,
    curve_type='kde'
)

# Update layout
fig_kde_only.update_layout(
    title_text='Smoothed Density Approximations (KDE) of Accuracies',
    xaxis_title='Accuracy',
    yaxis_title='Density'
)

fig_kde_only.show()

In [9]:
domains = ['Abduction', 'Deduction', 'Induction', 'MIX (A-D-I)']

# --- Create the combined box plot ---
fig_box_all = px.box(
    df,
    x='Domain',   # Puts each domain on the x-axis
    y='Accuracy', # Shows the accuracy distribution on the y-axis
    color='Domain', # Colors each box differently
    points='all',
    title='Distribution of Accuracies for All Domains'
)
fig_box_all.show()
fig_box_all.write_image("accuracies_distribution.png", scale=3)

In [12]:
# --- Generate the Final Simplified Plot ---
r_overall, _ = pearsonr(combined_df['Accuracy_Abduction'], combined_df['Accuracy'])
r2_overall = r_overall**2

fig_combined_simple = px.scatter(
    combined_df,
    x='Accuracy_Abduction',
    y='Accuracy',
    hover_data=['Model', 'Method Used', 'Source', 'Domain_Other'],
    trendline='ols',
    labels={
        'Accuracy_Abduction': 'Abductive Accuracy',
        'Accuracy': 'Accuracy (Other Domains)'
    },
    title=f'Abductive Accuracy vs. Other Domains (R = {r_overall:.2f}, R² = {r2_overall:.2f})'
)

# Make the trendline thicker
fig_combined_simple.data[1].update(line=dict(width=4))
fig_combined_simple.show()
fig_combined_simple.write_image("abductive_vs_all_other_domains_with_line.png", scale=3)

In [14]:
# --- Create Legend Labels with R and R^2 for each Domain Pair ---
legend_labels = {}
for pair in combined_df['Domain Pair'].unique():
    subset = combined_df[combined_df['Domain Pair'] == pair]
    if len(subset) > 1:
        r, _ = pearsonr(subset['Accuracy_Abduction'], subset['Accuracy'])
        r2 = r**2
        legend_labels[pair] = f"{pair} (R={r:.2f}, R²={r2:.2f})"
    else:
        legend_labels[pair] = f"{pair} (Not enough data)"

combined_df['Legend Label'] = combined_df['Domain Pair'].map(legend_labels)

fig_combined_domain = px.scatter(
    combined_df,
    x='Accuracy_Abduction',
    y='Accuracy',
    color='Legend Label', # Color by the new domain-only labels
    hover_data=['Model', 'Method Used', 'Source'],
    # trendline='ols',
    labels={
        'Accuracy_Abduction': 'Abductive Accuracy',
        'Accuracy': 'Accuracy (Other Domains)',
        'Legend Label': 'Domain Pair'
    },
    title=f'Abductive Accuracy vs. Other Domains'
)
fig_combined_domain.show()
fig_combined_domain.write_image("abductive_vs_all_other_domains_with_legends.png", scale=3)

In [17]:
# --- Plot 1: Abduction vs. Deduction (Simplified) ---

r_overall_ded, _ = pearsonr(merged_ded['Accuracy_Abduction'], merged_ded['Accuracy_Deduction'])
r2_overall_ded = r_overall_ded**2

fig_ded_simple = px.scatter(
    merged_ded,
    x='Accuracy_Abduction',
    y='Accuracy_Deduction',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Deduction'],
    trendline='ols',
    title=f'Abduction vs. Deduction Accuracy (Correlation = {r_overall_ded:.2f}, R² = {r2_overall_ded:.2f})'
)
# Make the trendline thicker
fig_ded_simple.data[1].update(line=dict(width=4))
fig_ded_simple.show()
fig_ded_simple.write_image("abductive_vs_deduction_with_line.png", scale=3)

In [19]:
# --- Plot 2: Abduction vs. Induction (Simplified) ---

r_overall_ind, _ = pearsonr(merged_ind['Accuracy_Abduction'], merged_ind['Accuracy_Induction'])
r2_overall_ind = r_overall_ind**2

fig_ind_simple = px.scatter(
    merged_ind,
    x='Accuracy_Abduction',
    y='Accuracy_Induction',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Induction'],
    # trendline='ols',
    title=f'Abduction vs. Induction Accuracy (Correlation = {r_overall_ind:.2f}, R² = {r2_overall_ind:.2f})'
)
# Make the trendline thicker
# fig_ind_simple.data[1].update(line=dict(width=4))
fig_ind_simple.show()
fig_ind_simple.write_image("abductive_vs_induction.png", scale=3)

In [21]:
# --- Plot 3: Abduction vs. Mix (Simplified) ---

r_overall_mix, _ = pearsonr(merged_mix['Accuracy_Abduction'], merged_mix['Accuracy_Mix'])
r2_overall_mix = r_overall_mix**2

fig_mix_simple = px.scatter(
    merged_mix,
    x='Accuracy_Abduction',
    y='Accuracy_Mix',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Mix'],
    # trendline='ols',
    title=f'Abduction vs. Mix Accuracy (Correlation = {r_overall_mix:.2f}, R² = {r2_overall_mix:.2f})'
)
# Make the trendline thicker
# fig_mix_simple.data[1].update(line=dict(width=4))
fig_mix_simple.show()
fig_mix_simple.write_image("abductive_vs_mix.png", scale=3)

In [22]:
# --- Plot 1: Abduction vs. Deduction ---

merged_ded = pd.merge(df_abduction, df_deduction, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Deduction'))
merged_ded['Model Family'] = merged_ded['Model'].apply(get_model_family)

fig_ded = px.scatter(
    merged_ded,
    x='Accuracy_Abduction',
    y='Accuracy_Deduction',
    color='Model Family',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Deduction'],
    trendline='ols',
    title=f'Abduction vs. Deduction Accuracy'
)

# Add family-specific R2 annotations below the title
fig_ded.update_layout(margin=dict(t=140)) # Increase top margin to make space
annotations = []
x_positions = [0.01, 0.22, 0.43, 0.64, 0.85] # Horizontal positions for text

for i, family in enumerate(merged_ded['Model Family'].unique()):
    family_df = merged_ded[merged_ded['Model Family'] == family]
    if len(family_df) > 1:
        r, _ = pearsonr(family_df['Accuracy_Abduction'], family_df['Accuracy_Deduction'])
        r2 = r**2
        annotations.append(dict(
            x=x_positions[i], y=1.09, xref='paper', yref='paper',
            text=f'<b>{family} R: {r:.2f}, R²: {r2:.2f}</b>', showarrow=False,
            font=dict(color=px.colors.qualitative.Plotly[i], size=12),
            xanchor='left'
        ))
for annotation in annotations:
    fig_ded.add_annotation(annotation)
fig_ded.show()
fig_ded.write_image("abductive_vs_deduction_grouped_by_model_family.png", scale=3)

In [23]:
# --- Plot 2: Abduction vs. Induction ---

merged_ind = pd.merge(df_abduction, df_induction, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Induction'))
merged_ind['Model Family'] = merged_ind['Model'].apply(get_model_family)

fig_ind = px.scatter(
    merged_ind,
    x='Accuracy_Abduction',
    y='Accuracy_Induction',
    color='Model Family',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Induction'],
    trendline='ols',
    title=f'Abduction vs. Induction Accuracy'
)

fig_ind.update_layout(margin=dict(t=140))
annotations = []
for i, family in enumerate(merged_ind['Model Family'].unique()):
    family_df = merged_ind[merged_ind['Model Family'] == family]
    if len(family_df) > 1:
        r, _ = pearsonr(family_df['Accuracy_Abduction'], family_df['Accuracy_Induction'])
        r2 = r**2
        annotations.append(dict(
            x=x_positions[i], y=1.09, xref='paper', yref='paper',
            text=f'<b>{family} R: {r:.2f}, R²: {r2:.2f}</b>', showarrow=False,
            font=dict(color=px.colors.qualitative.Plotly[i], size=12),
            xanchor='left'
        ))
for annotation in annotations:
    fig_ind.add_annotation(annotation)
fig_ind.show()
fig_ind.write_image("abductive_vs_induction_grouped_by_model_family.png", scale=3)

In [24]:
# --- Plot 3: Abduction vs. Mix ---

merged_mix = pd.merge(df_abduction, df_mix, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Mix'))
merged_mix['Model Family'] = merged_mix['Model'].apply(get_model_family)

fig_mix = px.scatter(
    merged_mix,
    x='Accuracy_Abduction',
    y='Accuracy_Mix',
    color='Model Family',
    hover_data=['Model', 'Method Used', 'Source', 'Test Dataset_Abduction', 'Test Dataset_Mix'],
    trendline='ols',
    title=f'Abduction vs. Mix Accuracy'
)

fig_mix.update_layout(margin=dict(t=140))
annotations = []
for i, family in enumerate(merged_mix['Model Family'].unique()):
    family_df = merged_mix[merged_mix['Model Family'] == family]
    if len(family_df) > 1:
        r, _ = pearsonr(family_df['Accuracy_Abduction'], family_df['Accuracy_Mix'])
        r2 = r**2
        annotations.append(dict(
            x=x_positions[i], y=1.09, xref='paper', yref='paper',
            text=f'<b>{family} R: {r:.2f}, R²: {r2:.2f}</b>', showarrow=False,
            font=dict(color=px.colors.qualitative.Plotly[i], size=12),
            xanchor='left'
        ))
for annotation in annotations:
    fig_mix.add_annotation(annotation)
fig_mix.show()
fig_mix.write_image("abductive_vs_mix_grouped_by_model_family.png", scale=3)

In [27]:
# 1. DEFINE A TRUNCATION LENGTH
TRUNCATE_AT = 8

# Create new columns with the truncated names
short_ds_ab = merged_ded['Test Dataset_Abduction'].str[:TRUNCATE_AT]
short_ds_de = merged_ded['Test Dataset_Deduction'].str[:TRUNCATE_AT]

# Combine the truncated names for the legend
merged_ded['Short Pair'] = short_ds_ab + " & " + short_ds_de

# 2. CREATE LEGEND LABELS USING THE TRUNCATED NAMES
legend_labels_ded = {}
for pair in merged_ded['Short Pair'].unique():
    subset = merged_ded[merged_ded['Short Pair'] == pair]
    if len(subset) > 1:
        r, _ = pearsonr(subset['Accuracy_Abduction'], subset['Accuracy_Deduction'])
        r2 = r**2
        legend_labels_ded[pair] = f"{pair} (R={r:.2f}, R²={r2:.2f})"
    else:
        legend_labels_ded[pair] = pair
merged_ded['Legend Label'] = merged_ded['Short Pair'].map(legend_labels_ded)

# 3. CREATE THE PLOT
fig_ded_ds = px.scatter(
    merged_ded,
    x='Accuracy_Abduction',
    y='Accuracy_Deduction',
    color='Legend Label',
    # Add the original, full names to the hover data for clarity
    hover_data={
        'Model': True,
        'Method Used': True,
        'Source': True,
        'Legend Label': False, # Hide the truncated label from hover
        'Test Dataset_Abduction': True,
        'Test Dataset_Deduction': True
    },
    trendline='ols',
    title='Abduction vs. Deduction Accuracy'
)

# 4. UPDATE LAYOUT TO MOVE LEGEND (optional, but recommended)
fig_ded_ds.update_layout(
    legend_title_text='Dataset Pair (Truncated)',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5
    )
)

fig_ded_ds.show()

In [40]:
# --- Plot 2: Abduction vs. Induction ---

merged_ind = pd.merge(df_abduction, df_induction, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Induction'))
merged_ind['Dataset Pair'] = merged_ind['Test Dataset_Abduction'] + " & " + merged_ind['Test Dataset_Induction']

legend_labels_ind = {}
for pair in merged_ind['Dataset Pair'].unique():
    subset = merged_ind[merged_ind['Dataset Pair'] == pair]
    if len(subset) > 1:
        r, _ = pearsonr(subset['Accuracy_Abduction'], subset['Accuracy_Induction'])
        r2 = r**2
        legend_labels_ind[pair] = f"{pair} (R={r:.2f}, R²={r2:.2f})"
    else:
        legend_labels_ind[pair] = f"{pair} (Not enough data)"
merged_ind['Legend Label'] = merged_ind['Dataset Pair'].map(legend_labels_ind)

fig_ind_ds = px.scatter(
    merged_ind,
    x='Accuracy_Abduction',
    y='Accuracy_Induction',
    color='Legend Label',
    hover_data=['Model', 'Method Used', 'Source'],
    trendline='ols',
    title=f'Abduction vs. Induction Accuracy'
)
fig_ind_ds.show()

In [28]:
# --- Plot 3: Abduction vs. Mix ---

merged_mix = pd.merge(df_abduction, df_mix, on=['Model', 'Method Used', 'Source'], suffixes=('_Abduction', '_Mix'))
merged_mix['Dataset Pair'] = merged_mix['Test Dataset_Abduction'] + " & " + merged_mix['Test Dataset_Mix']

legend_labels_mix = {}
for pair in merged_mix['Dataset Pair'].unique():
    subset = merged_mix[merged_mix['Dataset Pair'] == pair]
    if len(subset) > 1:
        r, _ = pearsonr(subset['Accuracy_Abduction'], subset['Accuracy_Mix'])
        r2 = r**2
        legend_labels_mix[pair] = f"{pair} (R={r:.2f}, R²={r2:.2f})"
    else:
        legend_labels_mix[pair] = f"{pair} (Not enough data)"
merged_mix['Legend Label'] = merged_mix['Dataset Pair'].map(legend_labels_mix)

fig_mix_ds = px.scatter(
    merged_mix,
    x='Accuracy_Abduction',
    y='Accuracy_Mix',
    color='Legend Label',
    hover_data=['Model', 'Method Used', 'Source'],
    trendline='ols',
    title=f'Abduction vs. Mix Accuracy'
)

fig_mix_ds.show()
fig_mix_ds.write_image("abductive_vs_mix_grouped_by_dataset_pair.png", scale=3)